# Comparação de Qualidade — 4 Dimensões: Texto, Tabelas, Equações, Gráficos

**Objetivo:** análise final agregada que consolida os outputs dos notebooks N1–N4 e responde "quanto os métodos concordam entre si em cada dimensão?".

**Dimensões avaliadas:**

1. **Texto** — concordância via Jaccard entre Chandra, GLM-OCR e LlamaParse.
2. **Tabelas** — idem.
3. **Equações** — idem (formatos diferentes: MathML do Chandra vs LaTeX do GLM/Llama).
4. **Gráficos** — duas partes:
   - **Quantidade detectada** (todos os métodos contra GT).
   - **Conteúdo textual** dos gráficos (com ChartVLM incluído como 5º método nesta dimensão).

**Sem GT absoluto** para texto/tabelas/equações — usa concordância pareada (Jaccard de palavras). A única métrica que usa GT manual é a quantidade de gráficos detectados.

> **Pré-requisito:** os notebooks N1–N4 rodaram e geraram outputs em `/MyDrive/Chandra2/output/`.


## 1. Setup

In [ ]:
!pip install -q pandas matplotlib seaborn beautifulsoup4

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

OUTPUT_ROOT = Path('/content/drive/MyDrive/Chandra2/output')
CHANDRA_DIR = OUTPUT_ROOT / 'chandra'
GLM_DIR     = OUTPUT_ROOT / 'glm_ocr'
LLAMA_DIR   = OUTPUT_ROOT / 'llamaparse'
CHARTVLM_OUT= OUTPUT_ROOT / 'chartvlm_results'
TEXT_OUT    = OUTPUT_ROOT / 'text_comparison'
TEXT_OUT.mkdir(parents=True, exist_ok=True)

GT_CSV = OUTPUT_ROOT / 'avaliacao_manual.csv'

TARGET_STEMS = [
    'W3132421076',
    'Mattos(2018)-IA 163 - Artigo Citros-Cafe - Fernanda Bochi Dos Santos',
    'W3110745114',
    'W4309496579',
    'Yamane(2022)-horticulturae-08-01126 - Janaina Lais Pacheco Lara Morandin (1)',
    'W3138442591',
    'grafico de radar',
    'W4323846702',
    'Mattos(2018)-TodaFruta2 - Fernanda Bochi Dos Santos',
    'W3216639369',
]
print(f"PDFs alvo: {len(TARGET_STEMS)}")

## 2. Extratores — texto, tabelas, equações, gráficos

Cada método produz formato diferente. Aqui normalizamos tudo em 5 componentes.

In [ ]:
import re, json
from bs4 import BeautifulSoup

def make_extract(text='', tables=None, equations=None, graphs_text=None, n_graphs=0):
    return {
        'text':        text or '',
        'tables':      tables or [],
        'equations':   equations or [],
        'graphs_text': graphs_text or [],   # descrições textuais de gráficos
        'n_graphs':    n_graphs,
    }

# ----- Chandra -----
def extract_chandra(stem):
    sub = CHANDRA_DIR / stem
    html_p = sub / f"{stem}.html"
    if not html_p.exists():
        return make_extract()
    raw_html = html_p.read_text(encoding='utf-8')
    soup = BeautifulSoup(raw_html, 'html.parser')

    tables = [t.get_text(' ', strip=True) for t in soup.find_all('table')]
    for t in soup.find_all('table'): t.decompose()

    equations = [m.get_text(strip=True) for m in soup.find_all('math')]
    for m in soup.find_all('math'): m.decompose()

    # Gráficos: cada <img> + texto do parágrafo seguinte (se houver)
    imgs = soup.find_all('img')
    n_graphs = len(imgs)
    graphs_text = []
    for img in imgs:
        nxt = img.find_next_sibling('p')
        if nxt:
            graphs_text.append(nxt.get_text(' ', strip=True))
        alt = img.get('alt', '').strip()
        if alt: graphs_text.append(alt)
    for img in imgs: img.decompose()

    text = soup.get_text(' ', strip=True)
    return make_extract(text, tables, equations, graphs_text, n_graphs)

# ----- GLM-OCR -----
def extract_glm(stem):
    md_p = GLM_DIR / stem / 'output.md'
    if not md_p.exists():
        return make_extract()
    raw = md_p.read_text(encoding='utf-8')

    md_tables = re.findall(r'(?:^\|[^\n]+\|\n){2,}', raw, flags=re.MULTILINE)
    html_tables = re.findall(r'<table[^>]*>[\s\S]*?</table>', raw, flags=re.IGNORECASE)
    tables = md_tables + html_tables

    equations = (re.findall(r'\$\$[^$]+\$\$', raw) +
                 re.findall(r'\\\[[\s\S]+?\\\]', raw))

    # Gráficos: ![alt](url) - deduplica pelo par (alt, url)
    graph_matches = re.findall(r'!\[([^\]]*)\]\(([^)]+)\)', raw)
    unique_graphs = set((a.strip().lower(), u.strip().lower()) for a, u in graph_matches)
    n_graphs = len(unique_graphs)
    graphs_text = [a for a, _ in graph_matches if a.strip()]

    cleaned = raw
    for t in tables:    cleaned = cleaned.replace(t, ' ')
    for e in equations: cleaned = cleaned.replace(e, ' ')
    cleaned = re.sub(r'!\[[^\]]*\]\([^)]+\)', ' ', cleaned)
    return make_extract(cleaned, tables, equations, graphs_text, n_graphs)

# ----- LlamaParse -----
def extract_llamaparse(stem):
    md_p = LLAMA_DIR / stem / 'output.md'
    if not md_p.exists():
        return make_extract()
    raw = md_p.read_text(encoding='utf-8')

    md_tables = re.findall(r'(?:^\|[^\n]+\|\n){2,}', raw, flags=re.MULTILINE)
    html_tables = re.findall(r'<table[^>]*>[\s\S]*?</table>', raw, flags=re.IGNORECASE)
    tables = md_tables + html_tables

    equations = (re.findall(r'\$\$[^$]+\$\$', raw) +
                 re.findall(r'\\\[[\s\S]+?\\\]', raw))

    # Gráficos + blocos "Chart data:"
    graph_imgs = re.findall(r'!\[([^\]]*)\]\([^)]+\)', raw)
    n_graphs = len(graph_imgs)
    graphs_text = [g for g in graph_imgs if g.strip()]
    chart_blocks = re.findall(
        r'(?:Chart data|Chart description|Extracted from chart|Figure description)[:\s]+([^\n]+(?:\n[^\n]+){0,5})',
        raw, flags=re.IGNORECASE
    )
    graphs_text.extend(chart_blocks)

    cleaned = raw
    for t in tables:    cleaned = cleaned.replace(t, ' ')
    for e in equations: cleaned = cleaned.replace(e, ' ')
    cleaned = re.sub(r'!\[[^\]]*\]\([^)]+\)', ' ', cleaned)
    return make_extract(cleaned, tables, equations, graphs_text, n_graphs)

EXTRACTORS = {
    'chandra':    extract_chandra,
    'glm':        extract_glm,
    'llamaparse': extract_llamaparse,
}

extracted = {}
for stem in TARGET_STEMS:
    extracted[stem] = {m: fn(stem) for m, fn in EXTRACTORS.items()}

print(f"{'PDF':<48} {'método':<11} {'chars':>7} {'tab':>4} {'eq':>4} {'graf':>5}")
print("-" * 80)
for stem in TARGET_STEMS[:3]:
    for m in EXTRACTORS:
        d = extracted[stem][m]
        print(f"  {stem[:46]:<46} {m:<11} {len(d['text']):>7} "
              f"{len(d['tables']):>4} {len(d['equations']):>4} {d['n_graphs']:>5}")
print("...")

## 3. Estatísticas básicas — quanto cada método extraiu (4 dimensões)

In [ ]:
import pandas as pd

def word_count(s): return len(s.split())

stats_rows = []
for stem, methods in extracted.items():
    for method, parts in methods.items():
        stats_rows.append({
            'pdf': stem[:35],
            'método': method,
            'chars_texto':  len(parts['text']),
            'palavras':     word_count(parts['text']),
            'n_tabelas':    len(parts['tables']),
            'chars_tabela': sum(len(t) for t in parts['tables']),
            'n_equações':   len(parts['equations']),
            'chars_eq':     sum(len(e) for e in parts['equations']),
            'n_gráficos':   parts['n_graphs'],
            'chars_graf':   sum(len(g) for g in parts['graphs_text']),
        })
stats_df = pd.DataFrame(stats_rows)

print("--- Médias por método nos 10 PDFs ---")
print(stats_df.groupby('método').agg({
    'chars_texto':  'mean',
    'palavras':     'mean',
    'n_tabelas':    'mean',
    'n_equações':   'mean',
    'n_gráficos':   'mean',
    'chars_graf':   'mean',
}).round(1).to_string())

print("\n--- Totais por método (todos os 10 PDFs) ---")
print(stats_df.groupby('método').agg({
    'palavras':   'sum',
    'n_tabelas':  'sum',
    'n_equações': 'sum',
    'n_gráficos': 'sum',
}).to_string())

stats_df.to_csv(TEXT_OUT / 'estatisticas_basicas_4dim.csv', index=False)

## 4. Jaccard pareado — concordância entre métodos (texto, tabelas, equações)

In [ ]:
def normalize_words(s):
    if not s: return set()
    s = s.lower()
    tokens = re.findall(r'\b[a-záàâãéêíóôõúç0-9\\]+\b', s)
    STOP = {'a','o','e','de','da','do','das','dos','um','uma','para','com','no','na',
            'em','é','foi','que','as','os','se','ao','aos','à','às','por','the',
            'and','or','of','in','to','is','on','at','for','as','this','that','was','be','by'}
    return {t for t in tokens if len(t) >= 2 and t not in STOP}

def jaccard(a, b):
    if not a and not b: return 1.0
    if not a or not b:  return 0.0
    return len(a & b) / len(a | b)

METHODS = list(EXTRACTORS.keys())

def jaccard_matrix(stems, dimension):
    accum = {(a,b): [] for a in METHODS for b in METHODS}
    for stem in stems:
        words = {}
        for m in METHODS:
            parts = extracted[stem][m]
            if dimension == 'text':         content = parts['text']
            elif dimension == 'tables':     content = ' '.join(parts['tables'])
            elif dimension == 'equations':  content = ' '.join(parts['equations'])
            elif dimension == 'graphs':     content = ' '.join(parts['graphs_text'])
            words[m] = normalize_words(content)
        for a in METHODS:
            for b in METHODS:
                accum[(a,b)].append(jaccard(words[a], words[b]))

    matrix = pd.DataFrame(0.0, index=METHODS, columns=METHODS)
    for (a,b), vals in accum.items():
        matrix.at[a,b] = round(sum(vals)/len(vals), 3) if vals else 0.0
    return matrix

jacc_text = jaccard_matrix(TARGET_STEMS, 'text')
jacc_tab  = jaccard_matrix(TARGET_STEMS, 'tables')
jacc_eq   = jaccard_matrix(TARGET_STEMS, 'equations')

print("=== JACCARD MÉDIO — TEXTO ===\n")
print(jacc_text.to_string())
print("\n=== JACCARD MÉDIO — TABELAS ===\n")
print(jacc_tab.to_string())
print("\n=== JACCARD MÉDIO — EQUAÇÕES ===\n")
print(jacc_eq.to_string())

jacc_text.to_csv(TEXT_OUT / 'jaccard_texto.csv')
jacc_tab.to_csv(TEXT_OUT / 'jaccard_tabelas.csv')
jacc_eq.to_csv(TEXT_OUT / 'jaccard_equacoes.csv')

## 5. GRÁFICOS — Parte 1: quantidade detectada vs ground truth

Compara o número de gráficos detectados por cada método com o GT (`gt_graficos + gt_imagens`).

In [ ]:
import numpy as np

gt_df = pd.read_csv(GT_CSV).set_index('pdf')

common_pdfs = [s + '.pdf' for s in TARGET_STEMS if s + '.pdf' in gt_df.index]
gt_graf_img = [int(gt_df.loc[n, 'gt_graficos']) + int(gt_df.loc[n, 'gt_imagens']) for n in common_pdfs]

method_counts = {}
for m in METHODS:
    method_counts[m] = [extracted[Path(n).stem][m]['n_graphs'] for n in common_pdfs]

def metrics(detected, gt):
    detected, gt = np.array(detected), np.array(gt)
    err = detected - gt
    return {
        'soma_detectado': int(detected.sum()),
        'soma_GT':        int(gt.sum()),
        'MAE':            round(float(np.abs(err).mean()), 2),
        'viés':           round(float(err.mean()), 2),
        'perfeitos':      int((err == 0).sum()),
    }

graph_results = {m: metrics(method_counts[m], gt_graf_img) for m in METHODS}
graph_rank = pd.DataFrame(graph_results).T.sort_values('MAE')
print("=== GRÁFICOS — Quantidade detectada vs GT ===\n")
print(graph_rank.to_string())

graph_rank.to_csv(TEXT_OUT / 'graficos_quantidade.csv')

detail_rows = []
for i, name in enumerate(common_pdfs):
    row = {'pdf': name[:40], 'GT': gt_graf_img[i]}
    for m in METHODS:
        row[m] = method_counts[m][i]
    detail_rows.append(row)
detail_graph = pd.DataFrame(detail_rows)
print("\n=== Detalhe por PDF ===\n")
print(detail_graph.to_string(index=False))

## 6. GRÁFICOS — Parte 2: conteúdo textual

Apenas Chandra (parágrafos pós-`<img>` + alt-text) e LlamaParse (`extract_charts=True`) produzem texto sobre gráficos de forma significativa. GLM-OCR usa alt-text vazio na maioria.

**Adicionalmente**, lê o output do **ChartVLM** (`chartvlm_classifications.csv`) e adiciona como 5º método nesta comparação específica.

In [ ]:
# ----- Carrega ChartVLM (se existir) -----
chartvlm_csv = CHARTVLM_OUT / 'chartvlm_classifications.csv'
chartvlm_text_per_pdf = {}

if chartvlm_csv.exists():
    cv_df = pd.read_csv(chartvlm_csv)
    for stem in TARGET_STEMS:
        sub = cv_df[cv_df['pdf'] == stem]
        texts = []
        for _, row in sub.iterrows():
            parts_l = [str(row.get('tipo_predito','')),
                       str(row.get('titulo','')),
                       str(row.get('descricao',''))]
            texts.append(' '.join(p for p in parts_l if p and p.lower() != 'nan'))
        chartvlm_text_per_pdf[stem] = texts
    print(f"✓ ChartVLM carregado ({len(cv_df)} crops)")
else:
    print("⚠️  ChartVLM CSV não encontrado")
    chartvlm_text_per_pdf = {stem: [] for stem in TARGET_STEMS}

In [ ]:
GRAPH_METHODS = METHODS + (['chartvlm'] if chartvlm_csv.exists() else [])

def get_graph_text(stem, method):
    if method == 'chartvlm':
        return ' '.join(chartvlm_text_per_pdf.get(stem, []))
    return ' '.join(extracted[stem][method]['graphs_text'])

def graph_text_jaccard():
    accum = {(a,b): [] for a in GRAPH_METHODS for b in GRAPH_METHODS}
    for stem in TARGET_STEMS:
        words = {m: normalize_words(get_graph_text(stem, m)) for m in GRAPH_METHODS}
        for a in GRAPH_METHODS:
            for b in GRAPH_METHODS:
                if not words[a] and not words[b]:
                    continue
                accum[(a,b)].append(jaccard(words[a], words[b]))

    matrix = pd.DataFrame(0.0, index=GRAPH_METHODS, columns=GRAPH_METHODS)
    for (a,b), vals in accum.items():
        matrix.at[a,b] = round(sum(vals)/len(vals), 3) if vals else float('nan')
    return matrix

jacc_graphs = graph_text_jaccard()
print("=== JACCARD MÉDIO — TEXTO SOBRE GRÁFICOS ===\n")
print(jacc_graphs.to_string())
jacc_graphs.to_csv(TEXT_OUT / 'jaccard_graficos_texto.csv')

print("\n=== Cobertura: PDFs com texto de gráfico produzido ===\n")
for m in GRAPH_METHODS:
    cov = sum(1 for stem in TARGET_STEMS if get_graph_text(stem, m).strip())
    chars = sum(len(get_graph_text(stem, m)) for stem in TARGET_STEMS)
    print(f"  {m:<12}: {cov}/{len(TARGET_STEMS)} PDFs com texto | total {chars} chars")

## 7. Visualização — heatmaps das 4 dimensões

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 2, figsize=(16, 13))

for ax, mat, title in zip(
    axes.flat,
    [jacc_text, jacc_tab, jacc_eq, jacc_graphs],
    ['TEXTO', 'TABELAS', 'EQUAÇÕES', 'TEXTO DE GRÁFICOS']
):
    sns.heatmap(mat, annot=True, fmt='.2f', cmap='RdYlGn',
                vmin=0, vmax=1, square=True, ax=ax,
                cbar_kws={'label': 'Jaccard'})
    ax.set_title(f'Concordância — {title}', fontsize=12)

plt.suptitle('Concordância pareada entre métodos (Jaccard de palavras)',
             fontsize=14, y=1.00)
plt.tight_layout()
plt.savefig(TEXT_OUT / 'heatmaps_4dim.png', dpi=120, bbox_inches='tight')
plt.show()
print(f"Salvo: {TEXT_OUT/'heatmaps_4dim.png'}")

In [ ]:
# Gráfico de barras: quantidade de gráficos detectados vs GT
fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(common_pdfs))
n_methods = len(METHODS)
w = 0.8 / (n_methods + 1)

ax.bar(x - n_methods*w/2, gt_graf_img, w, label='GT', color='black')
for i, m in enumerate(METHODS):
    ax.bar(x + (i - n_methods/2 + 0.5)*w, method_counts[m], w, label=m)

ax.set_xticks(x)
ax.set_xticklabels([n.replace('.pdf','')[:18] for n in common_pdfs],
                   rotation=45, ha='right', fontsize=8)
ax.set_ylabel('# gráficos detectados')
ax.set_title('Quantidade de gráficos por método vs ground truth')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(TEXT_OUT / 'graficos_quantidade_por_pdf.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Concordância média de cada método — todas as dimensões

In [ ]:
def avg_concordance(matrix, methods):
    rows = []
    for m in methods:
        others = [matrix.at[m, x] for x in methods if x != m and not pd.isna(matrix.at[m, x])]
        avg = round(sum(others)/len(others), 3) if others else float('nan')
        rows.append({'método': m, 'média_jaccard': avg})
    return pd.DataFrame(rows).sort_values('média_jaccard', ascending=False)

print("=== CONCORDÂNCIA MÉDIA COM OS DEMAIS ===\n")
print("--- TEXTO ---")
print(avg_concordance(jacc_text, METHODS).to_string(index=False))
print("\n--- TABELAS ---")
print(avg_concordance(jacc_tab, METHODS).to_string(index=False))
print("\n--- EQUAÇÕES ---")
print(avg_concordance(jacc_eq, METHODS).to_string(index=False))
print("\n--- TEXTO DE GRÁFICOS ---")
print(avg_concordance(jacc_graphs, GRAPH_METHODS).to_string(index=False))

## 9. Como ler os resultados

**Quantidade de gráficos (seção 5):**
- **MAE baixo** = método bate com o GT.
- **Viés positivo** = super-detecta (falsos positivos).
- **Viés negativo** = sub-detecta.

**Texto / tabelas / equações / texto-de-gráficos (seções 4, 6, 8):**
- **Diagonal = 1.0** (cada método consigo mesmo) — ignore.
- **0.7+** = forte concordância (extração robusta).
- **0.4–0.7** = média (diferenças de tokenização/ordem).
- **<0.4** = baixa (investigar). Pode indicar que um método pula conteúdo OU adiciona ruído.

**Saídas no Drive (`/MyDrive/Chandra2/output/text_comparison/`):**
- `estatisticas_basicas_4dim.csv` — quantidade extraída por método em cada PDF.
- `jaccard_{texto,tabelas,equacoes,graficos_texto}.csv` — matrizes 3×3 (4×4 com ChartVLM).
- `graficos_quantidade.csv` — ranking de detecção quantitativa.
- `heatmaps_4dim.png` — figura para o relatório.
- `graficos_quantidade_por_pdf.png` — barras comparativas.

**Estas saídas alimentam diretamente:**
- Tabela "Estatísticas de extração por método" (Materiais e Métodos).
- Tabela "Acurácia de detecção de gráficos" (Resultados).
- Figura "Heatmap de concordância" (Resultados).
- Figura "Detecção por PDF" (Resultados).
